# Aleatoriedad algorítmica y complejidad de Kolmogorov

Este cuaderno explora empíricamente la noción de aleatoriedad algorítmica:
compresibilidad de cadenas, tests estadísticos, y relación con la entropía.

**Artículo relacionado:** `03/12-aleatoriedad-algoritmica.md`

In [ ]:
import math
import random
import zlib
import struct
import numpy as np
from collections import Counter

## 1. Complejidad de Kolmogorov aproximada por compresión

No podemos calcular $K(x)$ exactamente (es incomputable), pero `zlib.compress`
da una cota superior: $K(x) \leq |\text{compress}(x)| + c$.

In [ ]:
def approx_kolmogorov(s):
    """
    Aproxima K(x) comprimiendo con zlib.
    Devuelve longitud en bytes del texto comprimido.
    """
    if isinstance(s, str):
        s = s.encode('utf-8')
    return len(zlib.compress(s, level=9))


def approx_kolmogorov_bits(seq):
    """Aproxima K(x) para secuencia binaria (lista de 0s y 1s)."""
    # Empaquetar bits en bytes
    byte_data = bytearray()
    for i in range(0, len(seq), 8):
        chunk = seq[i:i+8]
        byte = sum(b << j for j, b in enumerate(chunk))
        byte_data.append(byte)
    return len(zlib.compress(bytes(byte_data), level=9)) * 8


# Cadenas de distinta complejidad
sequences = [
    ("Ceros",    [0] * 1000),
    ("Unos",     [1] * 1000),
    ("Alternas", [i % 2 for i in range(1000)]),
    ("Periodo3", [i % 3 == 0 for i in range(1000)]),
    ("Thue-M",   None),  # se genera abajo
    ("Aleatorio", [random.randint(0, 1) for _ in range(1000)]),
]

# Secuencia de Thue-Morse: t(n) = paridad del peso de Hamming de n
thue_morse = [bin(i).count('1') % 2 for i in range(1000)]
sequences[4] = ("Thue-M", thue_morse)

random.seed(42)
sequences[5] = ("Aleatorio", [random.randint(0, 1) for _ in range(1000)])

print(f"{'Secuencia':<12} {'Longitud':<10} {'K aprox (bits)':<18} {'K/n':<10} {'Entropía H':<12} {'Compresible'}")
print('-' * 75)
for name, seq in sequences:
    n = len(seq)
    k_approx = approx_kolmogorov_bits(seq)
    counts = Counter(seq)
    entropy = -sum((c/n) * math.log2(c/n) for c in counts.values() if c > 0)
    compressible = "SÍ" if k_approx < n * 0.9 else "NO"
    print(f"{name:<12} {n:<10} {k_approx:<18} {k_approx/n:<10.3f} {entropy:<12.4f} {compressible}")

## 2. El argumento de conteo: la mayoría de cadenas son incompresibles

In [ ]:
def fraction_compressible(n, c_savings, n_samples=10000):
    """
    Estima la fracción de cadenas de longitud n que se comprimen más de c_savings bits.
    """
    random.seed(42)
    compressible = 0
    for _ in range(n_samples):
        seq = [random.randint(0, 1) for _ in range(n)]
        k = approx_kolmogorov_bits(seq)
        if k < n - c_savings:
            compressible += 1
    return compressible / n_samples


print("Fracción de cadenas aleatorias de longitud n que se comprimen > 10 bits:")
print(f"(predicción teórica: ≤ 2^(-10) = {2**(-10):.6f})")  
print()
print(f"{'n':<8} {'Fracción compresible':<22} {'Pred. teórica'}")
print('-' * 45)
for n in [50, 100, 200, 500]:
    frac = fraction_compressible(n, c_savings=10, n_samples=5000)
    print(f"{n:<8} {frac:<22.4f} {2**(-10):.6f}")

print()
print("Conclusión: la vasta mayoría de cadenas son incompresibles (≈ aleatorias).")
print("Los datos reales (texto, audio, imagen) son compresibles porque tienen estructura.")

## 3. Tests estadísticos de aleatoriedad

In [ ]:
def test_frequency(seq):
    """Test de frecuencia: fracción de 1s debería ser ≈ 0.5."""
    n = len(seq)
    ones = sum(seq)
    freq = ones / n
    # Estadístico: |ones - n/2| / sqrt(n/4)
    z = abs(ones - n/2) / math.sqrt(n/4)
    return freq, z


def test_runs(seq):
    """Test de rachas: número de rachas (runs) consecutivos."""
    if len(seq) < 2:
        return 0, 0
    n = len(seq)
    pi = sum(seq) / n
    if pi == 0 or pi == 1:
        return 0, float('inf')
    runs = 1 + sum(1 for i in range(1, n) if seq[i] != seq[i-1])
    exp_runs = 2 * n * pi * (1 - pi) + 1
    var_runs = 2 * n * pi * (1 - pi) * (2 * pi * (1 - pi) - 1) if n > 1 else 1
    z = (runs - exp_runs) / math.sqrt(max(var_runs, 1e-10))
    return runs, abs(z)


def test_entropy(seq, block_size=8):
    """Test de entropía: entropía de bloques de tamaño block_size."""
    n = len(seq)
    blocks = []
    for i in range(0, n - block_size + 1, block_size):
        block = tuple(seq[i:i+block_size])
        blocks.append(block)
    if not blocks:
        return 0, 0
    counts = Counter(blocks)
    total = len(blocks)
    h = -sum((c/total) * math.log2(c/total) for c in counts.values())
    max_h = block_size  # entropía máxima para bloques de block_size bits
    return h, h / max_h


# Aplicar tests a las secuencias
random.seed(42)
test_sequences = {
    "Ceros": [0] * 1000,
    "Alternas": [i % 2 for i in range(1000)],
    "Thue-Morse": [bin(i).count('1') % 2 for i in range(1000)],
    "Aleatorio": [random.randint(0, 1) for _ in range(1000)],
    "π (aprox)": [int(d) for d in 
                  '14159265358979323846264338327950288419716939937510' * 20
                  if d in '01'],  # bits de π aproximado
}

print(f"{'Secuencia':<15} {'Freq(1s)':<12} {'Z-freq':<10} {'Z-runs':<10} {'H/Hmax'}")
print('-' * 60)
for name, seq in test_sequences.items():
    freq, z_freq = test_frequency(seq)
    runs, z_runs = test_runs(seq)
    h, h_ratio = test_entropy(seq, block_size=4)
    print(f"{name:<15} {freq:<12.4f} {z_freq:<10.3f} {z_runs:<10.3f} {h_ratio:.4f}")

print()
print("Z-scores: |Z| < 2 suele indicar que la secuencia pasa el test (nivel 5%).")
print("H/Hmax ≈ 1 indica entropía máxima (máxima aleatoriedad).")

## 4. La secuencia de Thue-Morse: ¿aleatoria o no?

In [ ]:
# La secuencia de Thue-Morse pasa tests simples de aleatoriedad
# pero tiene complejidad de Kolmogorov O(log n) — es muy compresible

thue_morse = [bin(i).count('1') % 2 for i in range(1024)]

# Compresión
k_approx = approx_kolmogorov_bits(thue_morse)
print(f"Thue-Morse (n=1024 bits):")
print(f"  K aproximado: {k_approx} bits (vs {1024} bits del original)")
print(f"  Ratio K/n: {k_approx/1024:.3f}")
print()
print("Regla de generación: t(n) = paridad_de_Hamming(n) = número de 1s en binario de n, mod 2")
print("Descripción en pseudocódigo: 'for n in 0..N: print(popcount(n) % 2)'")
print("→ Descripción O(log N) bits, aunque la secuencia pase algunos tests de aleatoriedad.")
print()

# La secuencia no es aleatoria de Martin-Löf porque existe una regularidad:
# t(2n) = t(n), t(2n+1) = 1 - t(n) → estructura fractal
print("Regularidad de Thue-Morse: t(2n) = t(n), t(2n+1) = 1-t(n)")
violations = sum(1 for n in range(1, 512)
                 if thue_morse[2*n] != thue_morse[n] or
                 thue_morse[2*n+1] != 1 - thue_morse[n])
print(f"Violaciones de la regla en n=1..511: {violations} (debe ser 0)")
print()
print("Conclusión: pasar tests simples NO es suficiente para ser aleatoria de Martin-Löf.")
print("La aleatoriedad algorítmica exige pasar TODOS los tests computables.")

## 5. Entropía de Shannon vs complejidad de Kolmogorov

In [ ]:
# Relación entre entropía de Shannon (propiedad de distribución)
# y K(x) (propiedad de cadena individual)

print("Relación empírica entre H(fuente) y K(muestra típica)/n:")
print()

random.seed(42)
for p in [0.1, 0.2, 0.3, 0.5, 0.7, 0.9]:
    n = 500
    # Generar muestra i.i.d. con P(1)=p
    seq = [1 if random.random() < p else 0 for _ in range(n)]
    
    # Entropía teórica
    h = -p*math.log2(p) - (1-p)*math.log2(1-p) if 0 < p < 1 else 0
    
    # Complejidad aproximada
    k_approx = approx_kolmogorov_bits(seq) / n
    
    print(f"P(1)={p:.1f}: H={h:.4f} bits/símbolo, K/n≈{k_approx:.4f}")

print()
print("Para muestras típicas, K(x_1...x_n)/n ≈ H(fuente) (por el AEP).")
print("La aproximación mejora con n más grande y con p más alejado de 0.5")
print("(donde hay menos varianza en el peso de Hamming).")